# Iteration2 Story PDF Text Extraction

This notebook is for extracting text from the story PDF files in `data/raw/iteration2/story`. The extracted text is cleaned and saved as processed data for the second filter, which checks whether the text is useful reading material.

The output includes:

- one cleaned `.txt` file for each PDF
- one `story_text_manifest.csv` file to record the source file, output file, word count, and extraction status

The main goal is to keep the story text and remove text that is not part of the actual reading material, such as page numbers, licence text, StoryWeaver information, and image attribution text.

## 1. Setup

In this step, I import the required libraries and set the main file paths. The notebook finds the `TP10_DS` project folder, then sets the raw PDF folder and the processed output folder.

For PDF text extraction, I use `pdftotext`, this avoids adding another package for this small extraction task.

In [1]:
from pathlib import Path
import csv
import re
import shutil
import subprocess


def find_project_root(start: Path | None = None) -> Path:
    """Find TP10_DS whether the notebook is run from the repo root or notebook folder."""
    start = (start or Path.cwd()).resolve()
    # Allow the notebook to run from either TP10_DS/ or TP10_DS/notebooks/iteration2/.
    for candidate in (start, *start.parents):
        if (candidate / "data/raw/iteration2/story").exists():
            return candidate
        nested = candidate / "TP10_DS"
        if (nested / "data/raw/iteration2/story").exists():
            return nested
    raise FileNotFoundError("Could not find TP10_DS/data/raw/iteration2/story")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data/raw/iteration2/story"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/iteration2"
MANIFEST_PATH = OUTPUT_DIR / "story_text_manifest.csv"

# Prefer pdftotext from PATH, with the known local Anaconda path as a fallback.
PDFTOTEXT = shutil.which("pdftotext") or "/opt/anaconda3/bin/pdftotext"
if not Path(PDFTOTEXT).exists():
    raise FileNotFoundError("pdftotext was not found. Install poppler or add pdftotext to PATH.")

# Create the processed output folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw PDF directory: {RAW_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"pdftotext: {PDFTOTEXT}")

Project root: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS
Raw PDF directory: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/raw/iteration2/story
Output directory: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/iteration2
pdftotext: /opt/anaconda3/bin/pdftotext


## 2. Discover PDFs

This step finds the PDF files inside `data/raw/iteration2/story`. I only use the story PDF files here, not the `StoryWeaverAttribution_*.txt` files.

The PDF files are stored in separate story folders, so I use `*/*.pdf` to find them. There should be 7 PDF files in this iteration.

In [2]:
# Each story is stored in its own folder, so one wildcard level is enough.
pdf_paths = sorted(RAW_DIR.glob("*/*.pdf"))
print(f"Found {len(pdf_paths)} PDF files")
for pdf_path in pdf_paths:
    print(f"- {pdf_path.relative_to(PROJECT_ROOT)}")

Found 7 PDF files
- data/raw/iteration2/story/12293-bed-time-stories/12293-bed-time-stories.pdf
- data/raw/iteration2/story/202-bunty-and-bubbly/202-bunty-and-bubbly.pdf
- data/raw/iteration2/story/250-do-and-don-t/250-do-and-don-t.pdf
- data/raw/iteration2/story/4084-the-birthday-party/4084-the-birthday-party.pdf
- data/raw/iteration2/story/6100-bath-time-for-chunnu-and-munnu/6100-bath-time-for-chunnu-and-munnu.pdf
- data/raw/iteration2/story/8218-sam-s-christmas-present/8218-sam-s-christmas-present.pdf
- data/raw/iteration2/story/927-the-royal-toothache/927-the-royal-toothache.pdf


## 3. Extraction and Cleaning Helpers

This section defines the helper functions used for extraction and cleaning.

- `extract_pdf_text()` uses `pdftotext -layout` to extract text from each PDF.
- `clean_extracted_text()` removes text that is not useful for the reading-material dataset, such as page numbers, licence text, Creative Commons links, StoryWeaver information, image attribution, and disclaimer text.
- `apply_known_pdf_text_corrections()` fixes a few extraction errors that were found after checking the output, for example `butter ies` should be `butterflies`.

The cleaning is kept simple. I keep the title, author information, illustrator/translator information, and the story text. I do not rewrite the story content.

One issue is that some PDF text looks correct on the page but is copied or extracted incorrectly. For example, the word `butterflies` was extracted as `butter ies`. I added a small correction list for these checked cases only.

In [3]:
# When one of these lines appears, the remaining text is usually license,
# attribution, or platform description rather than story reading content.
LICENSE_STOP_PATTERNS = [
    re.compile(r"^this book was made possible", re.IGNORECASE),
    re.compile(r"^images? attribution", re.IGNORECASE),
    re.compile(r"^disclaimer\s*:", re.IGNORECASE),
    re.compile(r"^page\s+\d+\s*:", re.IGNORECASE),
    re.compile(r"^some rights reserved\.?", re.IGNORECASE),
    re.compile(r"^this book is cc", re.IGNORECASE),
    re.compile(r"creativecommons\.org", re.IGNORECASE),
    re.compile(r"^pratham books goes digital", re.IGNORECASE),
]

# Standalone page numbers and page markers are layout noise from the PDF.
DROP_LINE_PATTERNS = [
    re.compile(r"^\d+\s*/\s*\d+$"),
    re.compile(r"^\d+$"),
]

KNOWN_TEXT_CORRECTIONS = {
    "butter ies": "butterflies",
    "in nite": "infinite",
    "dirt o Chunnu": "dirt off Chunnu",
}


def extract_pdf_text(pdf_path: Path) -> str:
    # Output to stdout by passing '-' as the final pdftotext argument.
    result = subprocess.run(
        [PDFTOTEXT, "-layout", str(pdf_path), "-"],
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or f"pdftotext failed with exit code {result.returncode}")
    return result.stdout


def clean_extracted_text(raw_text: str) -> str:
    # Convert PDF page breaks to normal newlines before line-based cleaning.
    raw_text = raw_text.replace("\f", "\n")
    cleaned_lines = []

    for raw_line in raw_text.splitlines():
        # Collapse spacing inside each line while preserving paragraph breaks.
        line = re.sub(r"\s+", " ", raw_line).strip()
        if not line:
            if cleaned_lines and cleaned_lines[-1] != "":
                cleaned_lines.append("")
            continue

        # Stop before license/attribution sections that are not reading content.
        if any(pattern.search(line) for pattern in LICENSE_STOP_PATTERNS):
            break

        # Drop lines that only contain page numbers such as '2/8' or '7/12'.
        if any(pattern.fullmatch(line) for pattern in DROP_LINE_PATTERNS):
            continue

        # Remove inline page markers if pdftotext joins them with nearby text.
        line = re.sub(r"\b\d+\s*/\s*\d+\b", "", line).strip()
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def apply_known_pdf_text_corrections(text: str) -> str:
    # These are targeted fixes for confirmed PDF font/ligature extraction errors.
    for wrong_text, corrected_text in KNOWN_TEXT_CORRECTIONS.items():
        text = text.replace(wrong_text, corrected_text)
    return text


def count_words(text: str) -> int:
    # Count simple English words, allowing apostrophes and hyphenated words.
    return len(re.findall(r"\b\w+(?:['-]\w+)?\b", text))


def preview(text: str, length: int = 300) -> str:
    return re.sub(r"\s+", " ", text).strip()[:length]

## 4. Process PDFs

This section processes all PDF files. For each PDF, the notebook does the following steps:

1. Extract the raw text using `pdftotext`.
2. Clean the extracted text.
3. Apply the small correction list for known PDF extraction problems.
4. Save the cleaned text to `data/processed/iteration2/{pdf_name}.txt`.
5. Add one row to the manifest file for tracking.

The `notes` column stores a short text preview when the extraction is successful. If there is an error, it stores the error message instead.

In [4]:
manifest_rows = []

for pdf_path in pdf_paths:
    output_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
    # Start every row as failed, then update it after successful extraction.
    row = {
        "source_pdf": str(pdf_path.relative_to(PROJECT_ROOT)),
        "output_txt": str(output_path.relative_to(PROJECT_ROOT)),
        "title_slug": pdf_path.stem,
        "char_count": 0,
        "word_count": 0,
        "extraction_status": "failed",
        "notes": "",
    }

    try:
        # Extract first, clean second, then repair confirmed PDF text glitches.
        raw_text = extract_pdf_text(pdf_path)
        cleaned_text = clean_extracted_text(raw_text)
        cleaned_text = apply_known_pdf_text_corrections(cleaned_text)

        if not cleaned_text:
            row["extraction_status"] = "empty_text"
            row["notes"] = "No text remained after extraction and cleaning"
        else:
            # Store one cleaned TXT file per PDF for easy training-data review.
            output_path.write_text(cleaned_text + "\n", encoding="utf-8")
            row["char_count"] = len(cleaned_text)
            row["word_count"] = count_words(cleaned_text)
            row["extraction_status"] = "success"
            row["notes"] = preview(cleaned_text)
    except Exception as exc:
        # Keep processing later PDFs even if one file fails.
        row["notes"] = str(exc)

    manifest_rows.append(row)

fieldnames = [
    "source_pdf",
    "output_txt",
    "title_slug",
    "char_count",
    "word_count",
    "extraction_status",
    "notes",
]

# The manifest is the index for all extracted text files.
with MANIFEST_PATH.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f"Wrote manifest: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(manifest_rows)}")
for row in manifest_rows:
    print(f"{row['extraction_status']:>10} | {row['word_count']:>4} words | {row['title_slug']}")

Wrote manifest: data/processed/iteration2/story_text_manifest.csv
Rows: 7
   success |   86 words | 12293-bed-time-stories
   success |  130 words | 202-bunty-and-bubbly
   success |  126 words | 250-do-and-don-t
   success |  163 words | 4084-the-birthday-party
   success |  193 words | 6100-bath-time-for-chunnu-and-munnu
   success |  269 words | 8218-sam-s-christmas-present
   success |  240 words | 927-the-royal-toothache


## 5. Validation Preview

This final step is used to quickly check the processed output.

- Check how many PDF files were processed successfully.
- Read the generated `.txt` files.
- Print the first 300 characters of each story so I can check whether the output is real story text, not empty text, licence text, or attribution text.

In [5]:
# Show a short preview for manual QA after the files are written.
successful_rows = [row for row in manifest_rows if row["extraction_status"] == "success"]
print(f"Successful extractions: {len(successful_rows)} / {len(manifest_rows)}")

for row in successful_rows:
    text_path = PROJECT_ROOT / row["output_txt"]
    text = text_path.read_text(encoding="utf-8")
    print("\n" + "=" * 80)
    print(row["title_slug"])
    print("-" * 80)
    print(preview(text))

Successful extractions: 7 / 7

12293-bed-time-stories
--------------------------------------------------------------------------------
Bed-Time Stories Author: Kanchan Bannerjee Illustrator: Harshvardhan Gantha My Amma tells me a story every night. There is one story about a frog which talks. There is another story about a king with a long beard. Sometimes it is about a flying elephant. In one story a fox and a tiger are friends. T

202-bunty-and-bubbly
--------------------------------------------------------------------------------
Bunty and Bubbly Author: Sorit Gupto Illustrator: Sorit Gupto Bunty loves to play with butterflies... ...and with birds. She loves to play with paper boats. She also likes to make sand castles. When Bunty goes back home, her mother asks her to clean up. But she refuses. "I hate soaps!" she screams. 

250-do-and-don-t
--------------------------------------------------------------------------------
Do and Don't Author: Radha HS Illustrator: Ruchi Shah Good Mo